# Inspect the full-band peak finder on an analytic map — PIRATE 1.5

       Select the `pirate-15-validation` environment as your notebook kernel.
       This generates an analytic S/N map using a real plan and explicit producer
       decoding metadata. It provides controlled peak-finder experiments; its
       approximate response does not measure raw-stream dedispersion sensitivity.
       Earlier multi-method derivations are preserved in `historical/`.

In [ ]:
from pathlib import Path
import os
import sys
import pirate_frb

# Select the validation environment as the notebook kernel before running.
REPO_ROOT = Path(os.environ.get('PIRATE_REPO',
    Path(pirate_frb.__file__).resolve().parents[1])).expanduser().resolve()
if Path(pirate_frb.__file__).resolve().parents[1] != REPO_ROOT:
    raise RuntimeError('The kernel imported another PIRATE checkout. Select the 1.5 environment and restart the kernel.')
sys.path.insert(0, str(REPO_ROOT))
print('PIRATE:', pirate_frb.__file__)

In [ ]:
GPU_DEVICE = 0
TREE_INDEX = 0
SNR_THRESHOLD = 10.0
DM_REACH = 8
WAIST_BINS = 1
EDGE_POLICY = 'exclude'
DM_REACH_VALUES = (1, 2, 4, 8, 16)
MAX_TABLE_ROWS = 50
TIME_WINDOW_S = None
DM_WINDOW = None
TIMING_WARMUP_RUNS = 3
TIMING_MEASURED_RUNS = 20

CONFIG_PATH = REPO_ROOT / 'configs/dedispersion/chord_sb2.yml'
XENGINE_METADATA_PATH = REPO_ROOT / 'configs/xengine_metadata.yml'
INJECTED_DM = 100.0
INJECTED_TOAS_S = (9.0, 9.1, 9.15)
INJECTED_SNRS = (45.0, 30.0, 20.0)
INJECTED_WIDTH_MS = 1.0  # Intrinsic Gaussian sigma; decoded filter width differs.
BASE_SEED = 12345
TRIAL = 0
SAVE_GENERATED_MAP = False
GENERATED_MAP_PATH = REPO_ROOT / 'peakfinder_tests/generated_fast_snrmap_pirate15.npz'

In [ ]:
import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
import peakfinder_tests.peakfinders as pf

if pf.METHODS != ('full_band_bowtie',):
    raise RuntimeError('This notebook expects the retained production full-band peak finder.')
cp.cuda.Device(GPU_DEVICE).use()

from peakfinder_tests.experiment_common import prepare_plan, tree_for_dm
from peakfinder_tests.fast_snrmap import FastSnrMapSimulator

config, xmd, plan, dcores = prepare_plan(
    str(CONFIG_PATH), str(XENGINE_METADATA_PATH), cuda_device_id=GPU_DEVICE)
TREE_INDEX = tree_for_dm(plan, INJECTED_DM)
time_sample_s, nt_in = float(config.time_sample_ms) / 1000, int(plan.nt_in)
reference_freq_mhz = float(np.asarray(xmd.get_channel_freq_edges())[0])
_, subbands, full_index = pf.enumerate_plan_subbands(plan, TREE_INDEX, dcores=dcores)
simulator = FastSnrMapSimulator(
    plan, TREE_INDEX, subbands[full_index], dcores=dcores, xp=cp,
    time_sample_s=time_sample_s, nt_in=nt_in, reference_freq_mhz=reference_freq_mhz,
    injected_dm=INJECTED_DM, injected_snr=INJECTED_SNRS[0],
    injected_width_s=INJECTED_WIDTH_MS / 1000, base_seed=BASE_SEED,
)
time_chunk_index = simulator.common_chunk_index(INJECTED_TOAS_S)
snr_gpu, argmax_gpu, time_chunk_index = simulator.generate_map(
    INJECTED_TOAS_S, injected_snrs=INJECTED_SNRS, trial=TRIAL,
    time_chunk_index=time_chunk_index)
display(simulator.diagnostics())
print('Actual cadence (ms):', 1000 * time_sample_s, 'Derived chunk:', time_chunk_index)

## Analytic-model limits

       The simulator chooses one frequency multiplet, profile and extra-DM state
       near the requested burst DM, and evaluates the legal fine-time states.
       It combines deterministic signals with a seeded correlated background.
       It does not reproduce a real dedisperser's maximization over every profile,
       sub-band and DM state. Use the saved-map notebook for actual dedisperser output.

## Full-band bowtie geometry

The tree sets the coarse pixel sizes:
$\Delta DM=(DM_{max}-DM_{min})/n_{DM}$ and
$\Delta t=t_{sample}\,n_{t,in}/n_{t,out}$.
For a competitor at $\delta DM$, residual dispersion gives
$\delta t(f)=4148.808\,\delta DM\,(f_{ref}^{-2}-f^{-2})$ seconds, with frequencies in MHz.
The two observing-band edges bound the bowtie. `DM_REACH` is a radius in coarse DM
bins and `WAIST_BINS` adds a discrete allowance along its ridge.

The mask below is the actual production footprint. Candidate coordinates are
decoded with the producer's Dcores and all four token bytes, including extra DM.

In [ ]:
geometry = pf.build_peakfinder_geometry(
    plan, TREE_INDEX, dcores=dcores, time_sample_s=time_sample_s,
    nt_in=nt_in, reference_freq_mhz=reference_freq_mhz,
    dm_reach=DM_REACH, waist_bins=WAIST_BINS,
)
mask = cp.asnumpy(geometry.full_band_footprint)
dm_radius, time_radius = (size // 2 for size in mask.shape)
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.imshow(mask, origin='lower', interpolation='nearest', aspect='auto',
          extent=(-time_radius - .5, time_radius + .5, -dm_radius - .5, dm_radius + .5))
ax.set(xlabel='Time offset (coarse bins)', ylabel='DM offset (coarse bins)',
       title='Production full-band bowtie')
plt.show()
print('Producer Dcores:', tuple(dcores), 'Encoding:', pf.ARGMAX_ENCODING)
print('Coarse DM step:', geometry.dm_step, 'Coarse time step (ms):', 1000 * geometry.time_step_s)

## Select and decode candidates

This is a single-map inspection. `EDGE_POLICY="exclude"` omits centres whose
footprint crosses the selected map boundary. The streaming grouper additionally
uses neighbouring chunks, startup provenance, and grouping windows; its final
event catalog can therefore differ from this list of peak-finder candidates.

In [ ]:
candidates = pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                              geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
decoded = pf.decode_candidates(
    plan, candidates, dcores=dcores, itree=TREE_INDEX,
    time_chunk_index=time_chunk_index, ntime=nt_in, time_sample_s=time_sample_s,
)
candidate_rows = [
    dict(idm=int(decoded['idm'][i]), itime=int(decoded['itime'][i]),
         snr=float(decoded['snr'][i]), dm=float(decoded['dm'][i]),
         toa_s=float(decoded['toa_ref_s'][i]), width_ms=1000 * float(decoded['width_s'][i]),
         freq_lo_MHz=float(decoded['freq_lo_MHz'][i]),
         freq_hi_MHz=float(decoded['freq_hi_MHz'][i]),
         token=f"0x{int(decoded['argmax_token'][i]):08x}")
    for i in np.argsort(-decoded['snr'])[:MAX_TABLE_ROWS]
]
print('Candidates:', len(candidates))
display(candidate_rows)

## Map and physical candidate coordinates

       Background axes label coarse map cells. Candidate markers use fully decoded
       sub-cell coordinates and may be displaced within their source pixel.

In [ ]:
tree = plan.trees[TREE_INDEX]
dm_edges = float(tree.dm_min) + np.arange(snr_gpu.shape[0] + 1) * geometry.dm_step
time_edges = time_chunk_index * nt_in * time_sample_s + np.arange(snr_gpu.shape[1] + 1) * geometry.time_step_s
snr_cpu = cp.asnumpy(snr_gpu)
finite = snr_cpu[np.isfinite(snr_cpu)]
if not finite.size:
    raise ValueError('The selected map contains no finite S/N values.')
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
image = ax.pcolormesh(time_edges, dm_edges, snr_cpu, shading='flat', cmap='viridis',
                     vmin=float(np.percentile(finite, 1)), vmax=float(np.max(finite)))
fig.colorbar(image, ax=ax, label='S/N')
ax.scatter(decoded['toa_ref_s'], decoded['dm'], facecolors='none', edgecolors='red',
           label='Decoded candidate', s=65)
if 'INJECTED_TOAS_S' in globals():
    ax.scatter(INJECTED_TOAS_S, np.full(len(INJECTED_TOAS_S), INJECTED_DM),
               marker='x', color='white', label='Injected burst')
ax.set(xlabel=f'Time since sequence start (s), reference {reference_freq_mhz:g} MHz',
       ylabel='DM (pc cm$^{-3}$)', title=f'Full-band peak finder: tree {TREE_INDEX}, chunk {time_chunk_index}')
if TIME_WINDOW_S is not None:
    ax.set_xlim(*TIME_WINDOW_S)
if DM_WINDOW is not None:
    ax.set_ylim(*DM_WINDOW)
ax.legend()
plt.show()

## Effect of changing the DM reach

In [ ]:
reach_rows = []
for reach in DM_REACH_VALUES:
    reach_geometry = pf.build_peakfinder_geometry(
        plan, TREE_INDEX, dcores=dcores, time_sample_s=time_sample_s,
        nt_in=nt_in, reference_freq_mhz=reference_freq_mhz,
        dm_reach=reach, waist_bins=WAIST_BINS,
    )
    found = pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                             reach_geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
    reach_rows.append(dict(dm_reach=reach, candidates=len(found),
                           footprint_shape=tuple(reach_geometry.full_band_footprint.shape)))
display(reach_rows)

## Local timing diagnostic

       These measurements cover only the displayed map and peak-finder call.
       They are not an online capacity estimate or a performance campaign.

In [ ]:
if TIMING_WARMUP_RUNS < 0 or TIMING_MEASURED_RUNS < 1:
    raise ValueError('Timing needs nonnegative warmup and at least one measurement.')
for _ in range(TIMING_WARMUP_RUNS):
    pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                     geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
gpu_times_ms = []
for _ in range(TIMING_MEASURED_RUNS):
    start, stop = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                     geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
    stop.record()
    stop.synchronize()
    gpu_times_ms.append(cp.cuda.get_elapsed_time(start, stop))
print(f'Per-map peak-finding GPU time: median {np.median(gpu_times_ms):.3f} ms, '
      f'{len(gpu_times_ms)} measurements. This excludes generation, decoding and grouping.')

## Optional analytic-map export

       The NPZ identifies itself as an analytic experiment, with the plan, producer
       Dcores and token encoding required to interpret it. It is a separate format
       from the raw-frame dedisperser's ASDF products. Existing files are preserved.

In [ ]:
if SAVE_GENERATED_MAP:
    # Exclusive creation prevents accidental replacement of a previous experiment.
    with Path(GENERATED_MAP_PATH).open('xb') as stream:
        np.savez_compressed(
            stream, format='pirate.analytic_snrmap', format_version=np.int64(1),
            out_max=cp.asnumpy(snr_gpu), out_argmax=cp.asnumpy(argmax_gpu),
            config_yaml=config.to_yaml_string(), plan_yaml=plan.to_yaml_string(),
            dcores=np.asarray(dcores, dtype=np.int64), argmax_encoding=pf.ARGMAX_ENCODING,
            tree_index=np.int64(TREE_INDEX), time_chunk_index=np.int64(time_chunk_index),
            injected_dm=INJECTED_DM, injected_toas_s=np.asarray(INJECTED_TOAS_S),
            injected_snrs=np.asarray(INJECTED_SNRS), injected_width_sigma_ms=INJECTED_WIDTH_MS,
            base_seed=np.int64(BASE_SEED), trial=np.int64(TRIAL),
        )
    print('Saved:', GENERATED_MAP_PATH)